# TFT — Walmart Sales Forecasting

In [ ]:
import subprocess, os, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pickle
import wandb

warnings.filterwarnings('ignore')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN  = 52
PRED_LEN = 39
N_PAST   = 2
N_FUTURE = 1
N_STATIC = 3
WANDB_PROJECT = 'walmart-sales-forecasting-project'
WANDB_ENTITY  = 'ashos22-free-university-of-tbilisi-'

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

print(f'Device: {DEVICE}')

## WandB შესვლა

In [ ]:
subprocess.run(['rm', '-f', '/root/.netrc'], capture_output=True)
os.environ['WANDB_API_KEY'] = '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'
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
print('WandB login OK')

## მონაცემების ჩატვირთვა

In [ ]:
DATA_PATH = '/kaggle/input/walmart-recruiting-store-sales-forecasting/'

train    = pd.read_csv(DATA_PATH + 'train.csv.zip')
test     = pd.read_csv(DATA_PATH + 'test.csv.zip')
stores   = pd.read_csv(DATA_PATH + 'stores.csv')
features = pd.read_csv(DATA_PATH + 'features.csv.zip')

print('train:   ', train.shape)
print('test:    ', test.shape)
print('stores:  ', stores.shape)
print('features:', features.shape)
train.head()

## მონაცემების გასუფთავება

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='TFT_Cleaning', group='TFT_Training', reinit=True
)

train_df = train.merge(stores, on='Store')
train_df = train_df.merge(features, on=['Store', 'Date'], suffixes=('', '_feat'))
train_df.drop(columns=['IsHoliday_feat'], inplace=True)

test_df = test.merge(stores, on='Store')
test_df = test_df.merge(features, on=['Store', 'Date'], suffixes=('', '_feat'))
test_df.drop(columns=['IsHoliday_feat'], inplace=True)

for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
    train_df[col] = train_df[col].fillna(0)
    test_df[col]  = test_df[col].fillna(0)

for col in ['CPI', 'Unemployment']:
    train_df[col] = train_df[col].ffill()
    test_df[col]  = test_df[col].ffill()

type_map = {'A': 0, 'B': 1, 'C': 2}
train_df['Type'] = train_df['Type'].map(type_map)
test_df['Type']  = test_df['Type'].map(type_map)

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date']  = pd.to_datetime(test_df['Date'])

wandb.log({
    'train_rows': len(train_df),
    'test_rows':  len(test_df),
    'null_train': int(train_df.isnull().sum().sum())
})
run.finish()
print(f'train_rows: {len(train_df)},  null: {train_df.isnull().sum().sum()}')

## Feature Engineering — სტატიკური ნიშნებით

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='TFT_Feature_Engineering', group='TFT_Training', reinit=True
)

size_min  = float(train_df['Size'].min())
size_max  = float(train_df['Size'].max())
dept_max  = float(train_df['Dept'].max())

series_info  = {}
test_holiday = {}

for (store, dept), grp in train_df.groupby(['Store', 'Dept']):
    grp   = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values.astype(np.float32)
    if len(sales) < SEQ_LEN + PRED_LEN:
        continue
    is_holiday = grp['IsHoliday'].values.astype(bool)
    mean = sales.mean()
    std  = sales.std() + 1e-8

    store_type = float(grp['Type'].iloc[0]) / 2.0
    store_size = (float(grp['Size'].iloc[0]) - size_min) / (size_max - size_min + 1e-8)
    dept_norm  = float(dept) / dept_max

    series_info[(store, dept)] = {
        'sales':      sales,
        'is_holiday': is_holiday,
        'mean':       mean,
        'std':        std,
        'dates':      grp['Date'].values,
        'static':     np.array([store_type, dept_norm, store_size], dtype=np.float32)
    }

for (store, dept), grp in test_df.groupby(['Store', 'Dept']):
    grp = grp.sort_values('Date')
    test_holiday[(store, dept)] = grp['IsHoliday'].values.astype(np.float32)

test_dates   = sorted(test_df['Date'].unique())
n_test_dates = len(test_dates)

wandb.config.update({
    'n_series':       len(series_info),
    'avg_series_len': float(np.mean([len(v['sales']) for v in series_info.values()])),
    'seq_len':        SEQ_LEN,
    'pred_len':       PRED_LEN,
    'n_test_dates':   n_test_dates,
    'n_past':         N_PAST,
    'n_future':       N_FUTURE,
    'n_static':       N_STATIC
})
run.finish()
print(f'n_series: {len(series_info)},  test_dates: {n_test_dates}')

## TFT მოდელის კლასები

In [ ]:
class GatedLinearUnit(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc   = nn.Linear(input_size, output_size)
        self.gate = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))


class GatedResidualNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()
        if output_size is None:
            output_size = input_size
        self.fc1     = nn.Linear(input_size, hidden_size)
        self.fc2     = nn.Linear(hidden_size, hidden_size)
        self.glu     = GatedLinearUnit(hidden_size, output_size)
        self.norm    = nn.LayerNorm(output_size)
        self.skip    = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.elu(self.fc1(x))
        h = self.dropout(F.elu(self.fc2(h)))
        h = self.glu(h)
        return self.norm(self.skip(x) + h)


class TFTModel(nn.Module):
    def __init__(self, n_past, n_future, n_static, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()

        self.static_grn  = GatedResidualNetwork(n_static, d_model, d_model, dropout)
        self.static_ctx  = nn.Linear(d_model, d_model)

        self.past_proj   = nn.Linear(n_past, d_model)
        self.past_grn    = GatedResidualNetwork(d_model, d_model, dropout=dropout)
        self.encoder     = nn.LSTM(d_model, d_model, batch_first=True)

        self.future_proj = nn.Linear(n_future, d_model)
        self.future_grn  = GatedResidualNetwork(d_model, d_model, dropout=dropout)
        self.decoder     = nn.LSTM(d_model, d_model, batch_first=True)

        self.attention   = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm   = nn.LayerNorm(d_model)

        self.output_grn  = GatedResidualNetwork(d_model, d_model, dropout=dropout)
        self.output_proj = nn.Linear(d_model, 1)

    def forward(self, x_past, x_future, x_static):
        static_ctx = self.static_grn(x_static)
        h0 = self.static_ctx(static_ctx).unsqueeze(0)
        c0 = torch.zeros_like(h0)

        past_emb            = self.past_grn(self.past_proj(x_past))
        enc_out, (hn, cn)   = self.encoder(past_emb, (h0, c0))

        future_emb          = self.future_grn(self.future_proj(x_future))
        dec_out, _          = self.decoder(future_emb, (hn, cn))

        attn_out, _         = self.attention(dec_out, enc_out, enc_out)
        dec_out             = self.attn_norm(dec_out + attn_out)

        return self.output_proj(self.output_grn(dec_out)).squeeze(-1)


print('TFT კლასები მზადაა')

## Dataset და WMAE გამოთვლა

In [ ]:
class WalmartTFTDataset(Dataset):
    def __init__(self, series_info, seq_len, pred_len, split='train'):
        self.X_past, self.X_future, self.X_static, self.y = [], [], [], []

        for key, info in series_info.items():
            sales = info['sales']
            n     = len(sales)
            if n < seq_len + 2 * pred_len:
                continue
            mean, std  = info['mean'], info['std']
            norm       = (sales - mean) / std
            is_hol     = info['is_holiday'].astype(np.float32)
            static     = info['static']
            val_start  = n - pred_len

            if split == 'train':
                for i in range(seq_len, val_start - pred_len + 1):
                    past   = np.stack([norm[i - seq_len:i], is_hol[i - seq_len:i]], axis=-1)
                    future = is_hol[i:i + pred_len].reshape(-1, 1)
                    self.X_past.append(past)
                    self.X_future.append(future)
                    self.X_static.append(static)
                    self.y.append(norm[i:i + pred_len])
            else:
                if val_start >= seq_len:
                    past   = np.stack([norm[val_start - seq_len:val_start], is_hol[val_start - seq_len:val_start]], axis=-1)
                    future = is_hol[val_start:n].reshape(-1, 1)
                    self.X_past.append(past)
                    self.X_future.append(future)
                    self.X_static.append(static)
                    self.y.append(norm[val_start:n])

        self.X_past   = torch.FloatTensor(np.array(self.X_past))
        self.X_future = torch.FloatTensor(np.array(self.X_future))
        self.X_static = torch.FloatTensor(np.array(self.X_static))
        self.y        = torch.FloatTensor(np.array(self.y))

    def __len__(self):          return len(self.X_past)
    def __getitem__(self, idx): return self.X_past[idx], self.X_future[idx], self.X_static[idx], self.y[idx]


def compute_val_wmae(model, series_info, seq_len, pred_len):
    model.eval()
    all_true, all_pred, all_hol = [], [], []
    with torch.no_grad():
        for key, info in series_info.items():
            sales, is_holiday = info['sales'], info['is_holiday']
            mean, std = info['mean'], info['std']
            n = len(sales)
            val_start = n - pred_len
            if val_start < seq_len:
                continue
            norm   = (sales - mean) / std
            is_hol = is_holiday.astype(np.float32)
            past   = np.stack([norm[val_start - seq_len:val_start], is_hol[val_start - seq_len:val_start]], axis=-1)
            future = is_hol[val_start:n].reshape(-1, 1)

            x_past   = torch.FloatTensor(past).unsqueeze(0).to(DEVICE)
            x_future = torch.FloatTensor(future).unsqueeze(0).to(DEVICE)
            x_static = torch.FloatTensor(info['static']).unsqueeze(0).to(DEVICE)

            pred = model(x_past, x_future, x_static).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)
            all_pred.extend(pred)
            all_true.extend(sales[val_start:val_start + pred_len])
            all_hol.extend(is_holiday[val_start:val_start + pred_len])

    return wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))


print('Dataset კლასი მზადაა')

## ტრენინგის ფუნქცია

In [ ]:
def train_tft(series_info, config, run_name):
    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name=run_name, group='TFT_Training',
        config=config, reinit=True
    )

    model = TFTModel(
        n_past=N_PAST, n_future=N_FUTURE, n_static=N_STATIC,
        d_model=config['d_model'],
        n_heads=config['n_heads'],
        dropout=config['dropout']
    ).to(DEVICE)

    train_ds     = WalmartTFTDataset(series_info, SEQ_LEN, PRED_LEN, 'train')
    val_ds       = WalmartTFTDataset(series_info, SEQ_LEN, PRED_LEN, 'val')
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'])

    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.MSELoss()

    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0

    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for X_past, X_future, X_static, y in train_loader:
            X_past, X_future, X_static, y = (
                X_past.to(DEVICE), X_future.to(DEVICE),
                X_static.to(DEVICE), y.to(DEVICE)
            )
            optimizer.zero_grad()
            loss = criterion(model(X_past, X_future, X_static), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_past, X_future, X_static, y in val_loader:
                X_past, X_future, X_static, y = (
                    X_past.to(DEVICE), X_future.to(DEVICE),
                    X_static.to(DEVICE), y.to(DEVICE)
                )
                val_loss += criterion(model(X_past, X_future, X_static), y).item()
        val_loss /= len(val_loader)

        wandb.log({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                break

    model.load_state_dict(best_state)
    wmae_val = compute_val_wmae(model, series_info, SEQ_LEN, PRED_LEN)
    wandb.log({'wmae_val': wmae_val, 'best_val_loss': best_val_loss})
    run.finish()
    return model, wmae_val


print('train_tft() მზადაა')

## Baseline ტრენინგი

In [ ]:
baseline_config = {
    'd_model':    64,
    'n_heads':    4,
    'dropout':    0.1,
    'lr':         0.001,
    'batch_size': 256,
    'epochs':     50,
    'patience':   10
}

baseline_model, baseline_wmae = train_tft(series_info, baseline_config, 'TFT_Baseline')
print(f'Baseline WMAE: {baseline_wmae:.4f}')

## Tuned ტრენინგი

In [ ]:
tuned_config = {
    'd_model':    128,
    'n_heads':    4,
    'dropout':    0.1,
    'lr':         0.0005,
    'batch_size': 128,
    'epochs':     60,
    'patience':   12
}

tuned_model, tuned_wmae = train_tft(series_info, tuned_config, 'TFT_Tuned')
print(f'Tuned WMAE: {tuned_wmae:.4f}')

## შედარება — საუკეთესო მოდელი

In [ ]:
scores = {
    'TFT_Baseline': (baseline_model, baseline_wmae, baseline_config),
    'TFT_Tuned':    (tuned_model,    tuned_wmae,    tuned_config)
}

best_name  = min(scores, key=lambda k: scores[k][1])
best_model, best_wmae, best_config = scores[best_name]

print(f'TFT_Baseline WMAE: {baseline_wmae:.4f}')
print(f'TFT_Tuned    WMAE: {tuned_wmae:.4f}')
print(f'საუკეთესო: {best_name}  →  WMAE {best_wmae:.4f}')

## Test Predictions და Submission

In [ ]:
def predict_test(model, series_info, test_holiday, test_df, seq_len, pred_len):
    model.eval()
    result = test_df.copy()
    result['Date'] = pd.to_datetime(result['Date'])
    result['Weekly_Sales'] = 0.0

    with torch.no_grad():
        for (store, dept), info in series_info.items():
            mask = (result['Store'] == store) & (result['Dept'] == dept)
            if mask.sum() == 0:
                continue
            sales     = info['sales']
            mean, std = info['mean'], info['std']
            if len(sales) < seq_len:
                continue
            norm      = (sales - mean) / std
            is_hol    = info['is_holiday'].astype(np.float32)

            fut_hol = test_holiday.get((store, dept), np.zeros(pred_len, dtype=np.float32))
            if len(fut_hol) < pred_len:
                fut_hol = np.pad(fut_hol, (0, pred_len - len(fut_hol)))
            fut_hol = fut_hol[:pred_len]

            past     = np.stack([norm[-seq_len:], is_hol[-seq_len:]], axis=-1)
            future   = fut_hol.reshape(-1, 1)

            x_past   = torch.FloatTensor(past).unsqueeze(0).to(DEVICE)
            x_future = torch.FloatTensor(future).unsqueeze(0).to(DEVICE)
            x_static = torch.FloatTensor(info['static']).unsqueeze(0).to(DEVICE)

            pred = model(x_past, x_future, x_static).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)

            dates_sorted = sorted(result.loc[mask, 'Date'].unique())
            date_to_idx  = {d: i for i, d in enumerate(dates_sorted)}
            for row_idx, row in result.loc[mask].iterrows():
                idx = date_to_idx.get(row['Date'])
                if idx is not None and idx < len(pred):
                    result.loc[row_idx, 'Weekly_Sales'] = pred[idx]

    return result


predictions_df = predict_test(best_model, series_info, test_holiday, test_df, SEQ_LEN, PRED_LEN)

submission = predictions_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission['Id']   = submission.apply(
    lambda r: f"{int(r['Store'])}_{int(r['Dept'])}_{r['Date']}", axis=1
)
submission = submission[['Id', 'Weekly_Sales']]
submission.to_csv('tft_submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
submission.head()

## WandB Artifact — საუკეთესო მოდელის შენახვა

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='TFT_Best_Pipeline', group='TFT_Training', reinit=True
)

torch.save(best_model.state_dict(), 'tft_model.pt')

with open('series_info.pkl', 'wb') as f:
    pickle.dump(series_info, f)

with open('tft_config.pkl', 'wb') as f:
    pickle.dump({
        'seq_len':   SEQ_LEN,
        'pred_len':  PRED_LEN,
        'n_past':    N_PAST,
        'n_future':  N_FUTURE,
        'n_static':  N_STATIC,
        'd_model':   best_config['d_model'],
        'n_heads':   best_config['n_heads'],
        'dropout':   best_config['dropout']
    }, f)

artifact = wandb.Artifact(
    name='tft-walmart-sales',
    type='model',
    metadata={'wmae_val': best_wmae, 'best_version': best_name}
)
artifact.add_file('tft_model.pt')
artifact.add_file('series_info.pkl')
artifact.add_file('tft_config.pkl')
run.log_artifact(artifact)

wandb.log({'wmae_val_best': best_wmae})
run.finish()

print(f'WandB Artifact: tft-walmart-sales')
print(f'საუკეთესო WMAE: {best_wmae:.4f} ({best_name})')